In [ ]:
import os
import time
import mne
import numpy as np
import pandas as pd

import bsl
from bsl import StreamPlayer, datasets
# from bsl.externals import pylsl  # distributed version of pylsl
from bsl.triggers import TriggerDef

import pylsl

import pickle

import math
import matplotlib
import matplotlib.pyplot as plt
from pythonosc.udp_client import SimpleUDPClient

### OSC Client Intialization

In [ ]:
# OSC client initialization
ip = "127.0.0.1"
port = 4545
client = SimpleUDPClient(ip, port)

### Load Pretrained Model

In [ ]:
# Load your pretrained model and scaler
model = pickle.load("model.pkl")
scaler = pickle.load("scaler.pkl")

## Analyse Signal

In [ ]:
#TODO: Might want to change the buffer size and window size
receiver = bsl.StreamReceiver(bufsize=10, winsize=10, stream_name=['eeg', 'fnirs_device_name'])

#TODO: this channels will need to be updated 
eeg_info = mne.create_info(sfreq=250, ch_names=['E1'], ch_types=['eeg'])
fnirs_info = mne.create_info(sfreq=10, ch_names=['HbO', 'HbR'], ch_types=['fnirs_cw_amplitude'])
#TODO: some preprocessing here with baseline correction and baseline alpha power to normalize scores 
ref_mean_score = 0 
ref_std_score = 0
while True:
    receiver.acquire()

    #TODO: change stream name
    eeg_data, _ = receiver.get_window(stream_name='igeb')
    eeg_data = np.nan_to_num(eeg_data)
    # EEG: drop timestamp, keep EEG channel
    eeg_raw = mne.io.RawArray(data=eeg_data[:, [False, True]].T, info=eeg_info)
    eeg_raw.filter(1, 30)
    eeg_raw.crop(tmin=9)
    psds, _ = mne.time_frequency.psd_welch(eeg_raw, fmin=8, fmax=12, n_fft=125)
    alpha_score = np.mean(psds)
    alpha_norm = (alpha_score - ref_mean_score) / ref_std_score / 2

    
    #TODO: change stream name
    fnirs_data, fnirs_ts = receiver.get_window(stream_name='fnirs_device')

    # fNIRS: drop timestamp, keep oxy/deoxy channels
    fnirs_raw = mne.io.RawArray(data=fnirs_data[:, [False, True, True]].T, info=fnirs_info)
    fnirs_data = np.nan_to_num(fnirs_data)
    # Assuming channel 0 = timestamp, 1 = HbO, 2 = HbR
    fnirs_raw = mne.io.RawArray(data=fnirs_data[:, [False, True, True]].T,
                                 info=fnirs_info)
    fnirs_raw.filter(0.01, 0.5)  # Hemodynamic response is slow
    fnirs_raw.crop(tmin=9)
    # Convert to concentration (requires MNE fnirs processing)
    # fnirs_conc = mne.preprocessing.nirs.beer_lambert_law(fnirs_raw)
    # Or just use raw amplitude features
    hbo_mean = fnirs_raw.get_data(picks=['HbO']).mean()
    hbr_mean = fnirs_raw.get_data(picks=['HbR']).mean()

    features = np.array([[alpha_norm, hbo_mean, hbr_mean]])
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)

## Send Signal Via Osc

In [ ]:

client.send_message("/in/prediction", float(prediction[0]))

## open code code

Key design decisions:
- Window alignment: Both use the same 10s window, so features are temporally aligned
- Feature vector: [alpha_power, hbo_mean, hbr_mean] — you'd likely expand this (multiple EEG frequency bands, HbO/HbR slope, etc.)
- Scaler: fNIRS units (µM) differ wildly from EEG alpha power, so StandardScaler fit offline is essential
- Rate: The combined loop runs at whatever rate you set (4 Hz like the original, or slower to match fNIRS)

In [ ]:
# Main loop
while True:
    receiver.acquire()

    # --- EEG stream ---
    eeg_data, _ = receiver.get_window(stream_name='igeb')
    eeg_data = np.nan_to_num(eeg_data)
    eeg_raw = mne.io.RawArray(data=eeg_data[:, [False, True]].T, info=eeg_info)
    eeg_raw.filter(1, 30)
    eeg_raw.crop(tmin=9)
    psds, _ = mne.time_frequency.psd_welch(eeg_raw, fmin=8, fmax=12, n_fft=125)
    alpha_score = np.mean(psds)
    alpha_norm = (alpha_score - ref_mean_score) / ref_std_score / 2

    # --- fNIRS stream ---
    fnirs_data, _ = receiver.get_window(stream_name='fnirs_stream_name')
    fnirs_data = np.nan_to_num(fnirs_data)
    # Assuming channel 0 = timestamp, 1 = HbO, 2 = HbR
    fnirs_raw = mne.io.RawArray(data=fnirs_data[:, [False, True, True]].T,
                                 info=fnirs_info)
    fnirs_raw.filter(0.01, 0.5)  # Hemodynamic response is slow
    fnirs_raw.crop(tmin=9)
    # Convert to concentration (requires MNE fnirs processing)
    # fnirs_conc = mne.preprocessing.nirs.beer_lambert_law(fnirs_raw)
    # Or just use raw amplitude features
    hbo_mean = fnirs_raw.get_data(picks=['HbO']).mean()
    hbr_mean = fnirs_raw.get_data(picks=['HbR']).mean()

    # --- Combine and predict ---
    features = np.array([[alpha_norm, hbo_mean, hbr_mean]])
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)

    # Send to VR/neuromore
    client.send_message("/in/prediction", float(prediction[0]))